# Section 4.3 Generating or Updating the Search Index

*Notes:* This notebook prepares the vector index used for semantic retrieval over chunk content. The index is built from the caption field and refreshed after content updates.

The detailed background of this code is in this blog:



In [ ]:
%sql
-- Change Data Feed lets Vector Search sync only what changed
ALTER TABLE video_ai.silver.video_chunk_content
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Ensure a non-null primary key (chunk_id from 4.1)
ALTER TABLE video_ai.silver.video_chunk_content
ALTER COLUMN chunk_id SET NOT NULL;

ALTER TABLE video_ai.silver.video_chunk_content
ADD CONSTRAINT pk_video_chunk_content PRIMARY KEY (chunk_id);

-- Comment: These DDL changes enable incremental vector sync and ensure each chunk has a stable primary key.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType

w = WorkspaceClient()

endpoint_name = "video_ai_vs_endpoint"

# Check if endpoint already exists before creating
existing = [ep for ep in w.vector_search_endpoints.list_endpoints() if ep.name == endpoint_name]
if existing:
    print(f"Endpoint '{endpoint_name}' already exists, state: {existing[0].endpoint_status.state}")
else:
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type=EndpointType.STANDARD,
    )
    print(f"Endpoint '{endpoint_name}' created")

In [ ]:
from databricks.sdk.service.vectorsearch import (
    VectorIndexType, PipelineType, DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn
)

index_name = "video_ai.silver.video_chunk_index"

# Check if index already exists before creating
existing = [idx for idx in w.vector_search_indexes.list_indexes(endpoint_name="video_ai_vs_endpoint") if idx.name == index_name]
if existing:
    print(f"Index '{index_name}' already exists")
else:
    index = w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name="video_ai_vs_endpoint",
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="video_ai.silver.video_chunk_content",
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="caption",
                    embedding_model_endpoint_name="databricks-gte-large-en"
                )
            ]
        )
    )
    print(f"Index '{index_name}' created")

In [ ]:
import time

index_name = "video_ai.silver.video_chunk_index"

# Wait for index to be ready before querying
for i in range(60):
    status = w.vector_search_indexes.get_index(index_name)
    if status.status.ready:
        break
    if i == 0:
        print(f"Waiting for index '{index_name}' to be ready...")
    time.sleep(10)

results = w.vector_search_indexes.query_index(
    index_name=index_name,
    query_text="How does Delta Lake maintain consistency?",
    columns=["video_id", "chunk_id", "start_time", "end_time", "caption", "topic"],
    num_results=5,
    # filters_json='{"category": "Data Engineering"}',   # optional metadata filter
)
for row in results.result.data_array:
    print(row)

In [ ]:
%sql
SELECT video_id, start_time, end_time, topic, caption, search_score
FROM vector_search(
  index => 'video_ai.silver.video_chunk_index',
  query_text => 'How does Delta Lake maintain consistency?',
  num_results => 5
);


In [ ]:
w.vector_search_indexes.sync_index(
    index_name="video_ai.silver.video_chunk_index",
)

In [ ]:
%sql
-- Sanity-check retrieval quality with a few known questions
SELECT caption, topic, start_time, search_score
FROM vector_search(
  index => 'video_ai.silver.video_chunk_index',
  query_text => 'transaction log commits',
  num_results => 3
);
